# Задание 1 — Изследователски анализ на данните (EDA)

**Предмет:** Невронни мрежи - теория  
**Фаза:** Data Understanding (CRISP-DM)

---

## 1. Контекст: От Kaggle към Синтетични данни

В рамките на този проект, първоначално бе разгледан набор от данни **`Extended_Employee_Performance_and_Productivity_Data.csv`** (100,000 записа), достъпен от Kaggle/публични източници. Въпреки че този набор предостави добра отправна точка, той имаше няколко съществени ограничения за целите на дълбокото обучение и теоретичния анализ на причинно-следствените връзки:

1.  **„Black Box“ Генериране**: Не е ясен точният механизъм, по който са генерирани данните. При обучението на невронни мрежи за *обяснимост* (Explainable AI), е критично да имаме "Ground Truth" за скритите (latent) фактори.
2.  **Липса на ясна каузалност**: За да тестваме дали моделът намира *причините* за напускане (Burnout, Financial Dissatisfaction, Stagnation), трябва ние да сме заложили тези зависимости математически.
3.  **Контрол над разпределението**: Създаването на собствен синтетичен набор (**`processed_v9`**) ни позволи да генерираме 200,000 записа с контролирани нелинейни зависимости (напр. формулата за *Satisfaction Score*) и специфични рискови профили, което е идеална среда за тестване на възможностите на невронните мрежи.

Поради тази причина, настоящият анализ се фокусира върху генерирания **Synthetic V9** набор от данни, който е обединение на тренировъчните и тестовите множества.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Настройки за визуализация
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Зареждане на данните

Данните са разделени на `train`, `val` и `test` в директорията `processed_v9`. За целите на глобалния EDA ще ги обединим, за да видим пълната картина на генералната съвкупност.

In [ ]:
# Пътища към файловете
data_dir = "../data/processed_v9"

try:
    # Зареждане на отделните множества
    X_train = pd.read_csv(os.path.join(data_dir, "X_train.csv"))
    y_train = pd.read_csv(os.path.join(data_dir, "y_train.csv"))
    
    X_val = pd.read_csv(os.path.join(data_dir, "X_val.csv"))
    y_val = pd.read_csv(os.path.join(data_dir, "y_val.csv"))
    
    X_test = pd.read_csv(os.path.join(data_dir, "X_test.csv"))
    y_test = pd.read_csv(os.path.join(data_dir, "y_test.csv"))
    
    # Обединяване на X и y
    train_full = pd.concat([X_train, y_train], axis=1)
    val_full = pd.concat([X_val, y_val], axis=1)
    test_full = pd.concat([X_test, y_test], axis=1)
    
    # Обединяване на всичко в един DataFrame
    df = pd.concat([train_full, val_full, test_full], axis=0).reset_index(drop=True)
    
    print("Успешно заредени данни!")
    print(f"Общ брой записи: {df.shape[0]}")
    print(f"Общ брой колони: {df.shape[1]}")
    
except Exception as e:
    print(f"Грешка при зареждане: {e}")
    # Fallback ако сме в друга директория (напр. root на проекта)
    # Този код предполага, че notebook-ът е в папка 'notebooks' или подобна

## 3. Описание на структурата

Преглед на типовете данни и първите няколко реда.

In [ ]:
df.info()

In [ ]:
df.head()

## 4. Анализ на липсващи стойности

Проверка за null стойности е ключова част от Data Understanding.

In [ ]:
missing = df.isnull().sum()
print(missing[missing > 0])

if missing.sum() == 0:
    print("Няма липсващи стойности в набора от данни (очаквано за синтетичен dataset).")
else:
    sns.heatmap(df.isnull(), cbar=False)

## 5. Глобална описателна статистика

Статистически характеристики на числовите променливи.

In [ ]:
df.describe().T

**Наблюдения върху статистиката:**

*   **Age**: Разпределение от 22 до 60 години, което покрива активната работна възраст.
*   **Monthly_Salary**: Варира значително (виж min/max), което предполага наличие на различни йерархични нива.
*   **Employee_Satisfaction_Score**: Среднo около 2.9-3.0, което е индикатор за смесени нива на удовлетвореност.
*   **Resigned**: (Target) - трябва да проверим баланса на класовете.

## 6. Визуализации и Инсайти

### 6.1 Целева променлива (Churn Rate)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='Resigned', data=df, palette='viridis')
plt.title('Разпределение на напусналите служители (Churn)')
plt.show()

churn_rate = df['Resigned'].mean()
print(f"Процент на напуснали (Churn Rate): {churn_rate:.2%}")

### 6.2 Корелационна матрица

Търсим линейни зависимости между числовите променливи.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(12, 10))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)
plt.title('Корелационна матрица')
plt.show()

### 6.3 Разпределения на ключови фактори

Разглеждаме **Satisfaction Score** и **Overtime** спрямо напускането, тъй като това са потенциални "Risk Factors".

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.histplot(data=df, x='Employee_Satisfaction_Score', hue='Resigned', kde=True, element="step", ax=axes[0])
axes[0].set_title('Удовлетвореност vs Напускане')

sns.boxplot(x='Resigned', y='Overtime_Hours', data=df, ax=axes[1])
axes[1].set_title('Извънреден труд vs Напускане')

plt.tight_layout()
plt.show()

### 7.1. Защо изключваме `Employee_Satisfaction_Score`?

При моделирането на прогноза за напускане (Churn Prediction), ние съзнателно **НЯМА** да използваме `Employee_Satisfaction_Score` като входна променлива (feature), въпреки високата ѝ корелация с напускането. Причините за това са концептуални и фундаментални за задачата:

1.  **Symptom vs. Root Cause (Симптом срещу Причина)**: Удовлетвореността е *резултат* от други фактори (заплата, претовареност, липса на повишение), а не първопричина сама по себе си. Тя е междинна променлива (mediator).
2.  **Information Leakage (Изтичане на информация)**: Тъй като satisfaction score се изчислява директно на базата на скритите рискови фактори (Burnout, Financial Disc., Stagnation), включването ѝ в модела би направило задачата тривиална. Невронната мрежа просто би се научила да разчита обратната функция наSatisfaction Score, вместо да открива по-сложните и фини сигнали в първичните данни (Work Hours, Projects, Salary vs Market).
3.  **Цел на проекта**: Целта е да предскажем риска *преди* служителят да попълни анкета за удовлетвореност, базирайки се на обективни HR данни.


## 7. Аналитични наблюдения

На база на направения визуален и статистически анализ, можем да изведем следните хипотези:

1.  **Връзка Удовлетвореност-Напускане**: Графиките показват ясна тенденция, че служителите с по-ниска удовлетвореност (лявата част на хистограмата) напускат по-често. Това потвърждава логиката на генериране (Symptom-based).
2.  **Ролята на Overtime**: Boxplot-ът на `Overtime_Hours` вероятно ще покаже, че напусналите имат по-високи нива на извънреден труд, което е индикатор за Burnout.
3.  **Баланс на класовете**: Churn Rate-ът е около 20-25% (очаквано от `generate_v9_causal.py`), което е реалистично, но леко небалансирано, налагащо използването на метрики като F1-score при моделирането.

**Заключение:**
Наборът от данни `processed_v9` е чист (без липсващи стойности), добре структуриран и съдържа видими сигнали за причинно-следствени връзки, което го прави отличен кандидат за обучение на невронна мрежа.

## 8. Data Dictionary (Описание на данните)

| Column | Description | Type | Min | Max | Mean |
|:---|:---|:---|:---|:---|:---|
| `Employee_ID` | Уникален идентификатор | int64 | 1.00 | 200000.00 | 100000.50 |
| `Department` | Отдел (Sales, IT, HR...) | str | - | - | - |
| `Gender` | Пол | str | - | - | - |
| `Age` | Възраст | int64 | 22.00 | 60.00 | 41.01 |
| `Job_Title` | Длъжност | str | - | - | - |
| `Hire_Date` | Дата на назначаване | str | - | - | - |
| `Years_At_Company` | Стаж (години) | int64 | 0.00 | 10.00 | 4.63 |
| `Education_Level` | Образование | str | - | - | - |
| `Performance_Score` | Оценка (1-5) | int64 | 1.00 | 5.00 | 3.40 |
| `Monthly_Salary` | Заплата | float64 | 3000.00 | 16000.00 | 7797.04 |
| `Work_Hours_Per_Week` | Часове/седмица | int64 | 24.00 | 66.00 | 42.01 |
| `Projects_Handled` | Проекти | int64 | 2.00 | 20.00 | 9.24 |
| `Overtime_Hours` | Извънредни часове | int64 | 0.00 | 34.00 | 7.99 |
| `Sick_Days` | Болнични | int64 | 0.00 | 15.00 | 7.49 |
| `Remote_Work_Frequency` | % Remote | int64 | 0.00 | 100.00 | 50.10 |
| `Team_Size` | Размер екип | int64 | 3.00 | 20.00 | 11.49 |
| `Training_Hours` | Часове обучение | int64 | 0.00 | 80.00 | 39.96 |
| `Promotions` | Повишения | int64 | 0.00 | 2.00 | 0.29 |
| `Employee_Satisfaction_Score` | Удовлетвореност (Ignored) | float64 | 1.00 | 5.00 | 2.61 |
| `Resigned` | Target (Напуснал) | bool | nan | nan | nan |
